# 🧠 LM World Model — Full Research Pipeline
### `Qwen2.5-1.5B` · Unsloth SFT · RL Simulation · Mechanistic Interpretability

> *"Do Language Models Construct Internal World Models?"*  
> — Tinevimbo Musingadi, HIT Research Project (2026)  
> Directly positioned against Meta FAIR's Code World Model (32B, 2025)

---

| Stage | Cells | What happens |
|---|---|---|
| **0. Setup** | 2 | Git clone, install Unsloth + deps, mount Google Drive |
| **1. Data** | 2 | Load 10k Phase 1 traces, visualise distributions |
| **2. SFT** | 3 | Load Qwen2.5-1.5B, fine-tune (Condition B), plot loss / Grokking |
| **3. Evaluation** | 2 | OA, Levenshtein, OOD gap, Temperature Sensitivity |
| **4. RL** | 1 | GRPO-style reward rollouts, reward distribution |
| **5. MechInterp** | 3 | Logit Lens · Linear Probe · Activation Patching |
| **6. Save** | 1 | Push everything to Drive, print final summary |

**Runtime:** T4 GPU (free Colab) — ~4-6 hrs for full SFT · ~30 min for eval + interp

> ⚡ **Quick tip:** Runtime → Change runtime type → T4 GPU before running

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 0A — Clone Repo & Mount Google Drive
# ════════════════════════════════════════════════════════════
import os, sys

# Sanity check: must be running on GPU
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(f'GPU: {gpu}')
if not gpu:
    raise RuntimeError('❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

# Clone repository (skip if already present from previous session)
REPO_DIR = '/content/lm-world-model'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/TinevimboMusingadi/lm-world-model.git {REPO_DIR}
else:
    print(f'✅ Repo already cloned, pulling latest changes...')
    !git -C {REPO_DIR} pull --ff-only

# Mount Drive for checkpoint persistence across Colab sessions
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_DIR = '/content/drive/MyDrive/lm-world-model'
for sub in ['checkpoints', 'results', 'interp', 'logs']:
    os.makedirs(f'{DRIVE_DIR}/{sub}', exist_ok=True)

# Set working directory so all relative paths (data/, training/) work
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

print(f'\n✅ Working directory: {os.getcwd()}')
print(f'✅ Drive: {DRIVE_DIR}')
print(f'✅ Data file present: {os.path.exists("data/splits/phase1_raw.jsonl")}')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 0B — Install Dependencies
# Unsloth: ~4× faster SFT + 60% less VRAM vs vanilla HF
# ════════════════════════════════════════════════════════════
# Unsloth must be installed first as it patches torch internals
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade --no-deps trl peft accelerate bitsandbytes
!pip install -q wandb jsonlines python-Levenshtein plotly scikit-learn

import torch, transformers
print(f'\n✅ PyTorch   : {torch.__version__}')
print(f'✅ Transformers: {transformers.__version__}')
print(f'✅ CUDA        : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'✅ VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 1A — Load & Inspect the Phase 1 Trace Dataset
# 10,000 Python execution traces with full state at each step
# ════════════════════════════════════════════════════════════
import json, re
import jsonlines
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_FILE = 'data/splits/phase1_raw.jsonl'
records = []
with jsonlines.open(DATA_FILE) as reader:
    for r in reader:
        records.append(r)

def count_states(trace_str):
    """Count lines in the trace that actually contain variable state."""
    return sum(1 for l in trace_str.split('\n') if l.strip().startswith('[line=') and 'var_' in l)

df = pd.DataFrame([{
    'program_id': r['program_id'],
    'split':      r['split'],
    'complexity': str(r['complexity']),  # string for plotly color
    'state_steps': count_states(r['execution_trace']),
    'code_lines':  len(r['code'].strip().split('\n')),
    'has_branch':  'if ' in r['code'],
    'has_mul':     ' * ' in r['code'],
    'output_len':  len(str(r['output'])),
} for r in records])

print(f'📊 Total records: {len(df):,}')
print(df['split'].value_counts().to_frame().T.to_string())
print(f'\nComplexity distribution:')
print(df['complexity'].value_counts().sort_index().to_string())

# ── Four summary plots in a 2×2 grid ──
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    'Execution Steps by Complexity', 'Dataset Split Sizes',
    'Code Length Distribution', 'Branch/Mul Instruction %'
])

for comp, color in [('1','#636EFA'),('2','#EF553B'),('3','#00CC96')]:
    sub = df[df['complexity']==comp]['state_steps']
    fig.add_trace(go.Histogram(x=sub, name=f'Complexity {comp}', marker_color=color, opacity=0.7, nbinsx=25), row=1, col=1)

split_counts = df['split'].value_counts().reset_index()
fig.add_trace(go.Bar(x=split_counts['split'], y=split_counts['count'], marker_color='#AB63FA', showlegend=False), row=1, col=2)

fig.add_trace(go.Histogram(x=df['code_lines'], marker_color='#FFA15A', nbinsx=20, showlegend=False), row=2, col=1)

inst_data = {'Has branch (if)': df['has_branch'].mean()*100, 'Has multiply': df['has_mul'].mean()*100}
fig.add_trace(go.Bar(x=list(inst_data.keys()), y=list(inst_data.values()), marker_color=['#19D3F3','#FF6692'], showlegend=False), row=2, col=2)

fig.update_layout(title='Phase 1 Dataset Overview', template='plotly_dark', height=600, barmode='overlay')
fig.show()

# Print a sample trace for visual inspection
print('\n━━━ Sample (complexity=2, with branch) ━━━')
sample_idx = df[(df['complexity']=='2') & df['has_branch']].index[0]
r = records[sample_idx]
print(f"CODE:\n{r['code']}")
print(f"\nTRACE:\n{r['execution_trace']}")
print(f"\nOUTPUT: {r['output']}")

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 1B — Define Prompt Format & Show Training Examples
# Condition A = output only | Condition B = full trace (MAIN)
# ════════════════════════════════════════════════════════════
SYSTEM_PROMPT = (
    "You are a neural computer that executes Python code. "
    "Given a program, produce the full execution trace showing the state of all variables "
    "after each line, then give the final printed output inside <o></o> tags."
)

# Alpaca prompt template — simple and battle-tested
ALPACA_TEMPLATE = (
    "Below is an instruction that describes a task, paired with an input that provides "
    "further context. Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n{output}"
)

def build_response(record, condition='B'):
    code_block  = f"<code>\n{record['code']}\n</code>"
    trace_block = f"<execution_trace>\n{record['execution_trace']}\n</execution_trace>"
    output_block= f"<o>{record['output']}</o>"
    if condition == 'A':
        return f"{code_block}\n\n{output_block}"
    else:  # B — full trace supervision
        return f"{code_block}\n\n{trace_block}\n\n{output_block}"

def format_for_sft(record, condition='B', eos=''):
    return ALPACA_TEMPLATE.format(
        instruction=SYSTEM_PROMPT,
        input=f"<code>\n{record['code']}\n</code>",
        output=build_response(record, condition)
    ) + eos

# Show side-by-side: Condition A vs B
ex = records[5]
print('══════ CONDITION A (output only — baseline) ══════')
print(format_for_sft(ex, 'A')[:300], '...')
print('\n══════ CONDITION B (full trace — main experiment) ══════')
print(format_for_sft(ex, 'B')[:500], '...')
print(f'\nEstimated tokens (Condition B): ~{len(format_for_sft(ex,"B"))//4}')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 2A — Load Qwen2.5-1.5B via Unsloth (4× faster SFT)
# Handles QLoRA, RoPE scaling & gradient checkpointing
# ════════════════════════════════════════════════════════════
from unsloth import FastLanguageModel
import torch

# ─── Config ───────────────────────────────────────────────
MODEL_NAME   = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_SEQ_LEN  = 1024   # Covers ~90% of our traces
LORA_R       = 16
CONDITION    = 'B'    # Full trace supervision
CHECKPOINT   = f'{DRIVE_DIR}/checkpoints/qwen1.5b_condB'

# ─── Check for existing checkpoint (resume across Colab sessions) ──
existing_ckpt = None
if os.path.exists(CHECKPOINT):
    import glob
    ckpt_dirs = sorted(glob.glob(f'{CHECKPOINT}/checkpoint-*'))
    if ckpt_dirs:
        existing_ckpt = ckpt_dirs[-1]
        print(f'♻️  Found checkpoint to resume from: {existing_ckpt}')
    elif os.path.exists(f'{CHECKPOINT}/adapter_config.json'):
        print(f'✅ Found completed checkpoint at {CHECKPOINT}')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,  # auto: bf16 on Ampere (A100), fp16 on T4
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

EOS = tokenizer.eos_token
print(f'\n✅ Model: {MODEL_NAME} | EOS: {repr(EOS)}')
print(f'   VRAM used after loading: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 2B — Prepare Dataset & Run SFT Training
# Uses Condition B: predict full execution trace + output
# ════════════════════════════════════════════════════════════
import wandb
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

# W&B (will prompt for API key on first run; skip with ctrl+c to disable)
try:
    wandb.init(
        project='lm-world-model',
        name=f'phase1-qwen1.5b-cond{CONDITION}-unsloth',
        config={'model': MODEL_NAME, 'condition': CONDITION,
                'lora_r': LORA_R, 'max_seq_len': MAX_SEQ_LEN},
        resume='allow'
    )
    REPORT_TO = 'wandb'
except Exception:
    REPORT_TO = 'none'
    print('W&B skipped — logging to console only')

# Build dataset — filter to train split, then format
# Pre-truncate each text so every seq fits in MAX_SEQ_LEN tokens.
# This prevents the Unsloth padding_free + max_seq_length ValueError.
train_recs = [r for r in records if r["split"] == "train"]

def truncate_to_tokens(text, max_tokens=MAX_SEQ_LEN):
    ids = tokenizer(
        text, truncation=True, max_length=max_tokens,
        add_special_tokens=False)["input_ids"]
    return tokenizer.decode(ids, skip_special_tokens=False)

trunc_texts = [truncate_to_tokens(format_for_sft(r, CONDITION, EOS)) for r in train_recs]
hf_dataset  = Dataset.from_dict({"text": trunc_texts})

print(f'Training samples: {len(hf_dataset):,}')
print(f'Example (first 400 chars):')
print(hf_dataset[0]['text'][:400], '...')

# ─── Train ───────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    args=SFTConfig(
        output_dir=CHECKPOINT,
        resume_from_checkpoint=existing_ckpt,  # Auto-resume if checkpoint exists
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,   # Effective batch = 16
        warmup_steps=50,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        save_strategy='steps',
        save_steps=200,
        save_total_limit=3,
        report_to=REPORT_TO,
        max_seq_length=MAX_SEQ_LEN,
        dataset_text_field='text',
        packing=True,            # Fix: Unsloth padding_free requires packing with max_seq_length
        packing_strategy='bfd',  # Best-Fit Decreasing — also ~20% faster
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
    ),
)

trainer_stats = trainer.train()
print(f'\n✅ Training complete!')
print(f'   Runtime : {trainer_stats.metrics["train_runtime"]/3600:.2f} hrs')
print(f'   Samples/s: {trainer_stats.metrics["train_samples_per_second"]:.2f}')
print(f'   VRAM peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 2C — Training Curve + Grokking Detection
# ════════════════════════════════════════════════════════════
import json
import plotly.graph_objects as go

log_history = trainer.state.log_history
# Persist training log to Drive immediately
with open(f'{DRIVE_DIR}/logs/training_log.json', 'w') as f:
    json.dump(log_history, f)

steps  = [x['step'] for x in log_history if 'loss' in x]
losses = [x['loss'] for x in log_history if 'loss' in x]
grad_norms = [x.get('grad_norm', None) for x in log_history if 'loss' in x]

# Detect potential Grokking: sudden loss drop after plateau
def detect_grokking(losses, window=5, threshold=0.3):
    drops = []
    for i in range(window, len(losses)):
        prev_avg = sum(losses[i-window:i]) / window
        if prev_avg - losses[i] > threshold * prev_avg:
            drops.append(i)
    return drops

grok_steps = [steps[i] for i in detect_grokking(losses)]

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    'Training Loss (Grokking Detector)', 'Gradient Norm Over Training'])

fig.add_trace(go.Scatter(x=steps, y=losses, mode='lines', name='Loss',
                          line=dict(color='#EF553B', width=2)), row=1, col=1)
# Running average
import pandas as pd
s = pd.Series(losses)
fig.add_trace(go.Scatter(x=steps, y=s.rolling(10, center=True).mean(),
                          mode='lines', name='Smoothed', line=dict(color='#FF6692', width=1, dash='dash')), row=1, col=1)
for gs in grok_steps:
    fig.add_vline(x=gs, line_dash='dot', line_color='yellow', row=1, col=1)

valid_gnorms = [(s, g) for s, g in zip(steps, grad_norms) if g is not None]
if valid_gnorms:
    gn_steps, gn_vals = zip(*valid_gnorms)
    fig.add_trace(go.Scatter(x=gn_steps, y=gn_vals, mode='lines', name='Grad Norm',
                              line=dict(color='#00CC96', width=2)), row=1, col=2)

fig.update_layout(title='SFT Training Dynamics', template='plotly_dark')
if grok_steps:
    fig.add_annotation(text=f'⚡ Possible grokking at step {grok_steps[0]}',
                        x=grok_steps[0], y=min(losses), showarrow=True, arrowhead=2, row=1, col=1)
fig.show()

final_loss = losses[-1] if losses else float('nan')
print(f'Final training loss: {final_loss:.4f}')
print(f'Grokking detected at steps: {grok_steps if grok_steps else "None"}')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 3A — Evaluation: OA, Levenshtein, FES, OOD Gap
# All metrics from the formal research proposal
# ════════════════════════════════════════════════════════════
from unsloth import FastLanguageModel
import Levenshtein

FastLanguageModel.for_inference(model)  # Unsloth 2× inference speed

def extract_output(text):
    m = re.search(r'<o>(.*?)</o>', text, re.DOTALL)
    return m.group(1).strip() if m else ''

def extract_trace(text):
    m = re.search(r'<execution_trace>(.*?)</execution_trace>', text, re.DOTALL)
    return m.group(1).strip() if m else ''

def first_error_step(pred_trace, true_trace):
    """Return the index of the first diverging trace line (FES metric)."""
    pred_lines = [l.strip() for l in pred_trace.split('\n') if l.strip()]
    true_lines = [l.strip() for l in true_trace.split('\n') if l.strip()]
    for i, (p, t) in enumerate(zip(pred_lines, true_lines)):
        if p != t:
            return i
    return len(true_lines)  # No error found

def evaluate_split(eval_records, temperature=0.0, n_samples=200, label='test_indist'):
    results, n = [], min(n_samples, len(eval_records))
    print(f'Evaluating {n} samples [{label} | T={temperature}]...')
    for i, r in enumerate(eval_records[:n]):
        if i % 25 == 0: print(f'  {i}/{n}...', end='', flush=True)

        # Build inference prompt (response blank — model fills it in)
        prompt = ALPACA_TEMPLATE.format(
            instruction=SYSTEM_PROMPT,
            input=f"<code>\n{r['code']}\n</code>",
            output=''  # ← intentionally empty
        )
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True,
                           max_length=MAX_SEQ_LEN).to('cuda')

        gen_kwargs = dict(
            max_new_tokens=512,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=(temperature > 0.0),
        )
        if temperature > 0.0:
            gen_kwargs['temperature'] = temperature

        with torch.no_grad():
            out = model.generate(**inputs, **gen_kwargs)

        generated = tokenizer.decode(
            out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        pred_output = extract_output(generated)
        pred_trace  = extract_trace(generated)
        true_trace  = r['execution_trace']

        # Normalise output comparison (strip whitespace + sign formatting)
        out_correct = (pred_output.strip().lstrip('0') == r['output'].strip().lstrip('0'))
        lev  = Levenshtein.distance(pred_trace, true_trace)
        fes  = first_error_step(pred_trace, true_trace)
        # Trace normalised edit distance (0=identical, 1=completely different)
        ned  = lev / max(len(true_trace), 1)

        results.append({
            'program_id':     r['program_id'],
            'complexity':     r['complexity'],
            'split':          label,
            'temperature':    temperature,
            'output_correct': out_correct,
            'levenshtein':    lev,
            'norm_edit_dist': ned,
            'first_error_step': fes,
            'pred_output':    pred_output,
            'true_output':    r['output'],
        })

    oa      = sum(r['output_correct'] for r in results) / len(results) * 100
    avg_lev = sum(r['levenshtein'] for r in results) / len(results)
    avg_fes = sum(r['first_error_step'] for r in results) / len(results)
    print(f'\n  OA={oa:.1f}% | Avg Lev={avg_lev:.0f} | Avg FES={avg_fes:.1f}')
    return results

# ─── Run all splits ─────────────────────────────────────────
val_recs    = [r for r in records if r['split'] == 'val']
indist_recs = [r for r in records if r['split'] == 'test_indist']
ood_recs    = [r for r in records if r['split'] == 'test_ood']
long_recs   = [r for r in records if r['split'] == 'test_long']

res_val    = evaluate_split(val_recs,    temperature=0.0, n_samples=100, label='val')
res_indist = evaluate_split(indist_recs, temperature=0.0, n_samples=100, label='test_indist')
res_ood    = evaluate_split(ood_recs,    temperature=0.0, n_samples=100, label='test_ood')
res_indist_t5 = evaluate_split(indist_recs, temperature=0.5, n_samples=50, label='test_indist')  # Temperature test

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 3B — Evaluation Visualizations
# The full set of research metrics from the proposal
# ════════════════════════════════════════════════════════════
all_results = res_val + res_indist + res_ood + res_indist_t5
df_res = pd.DataFrame(all_results)
df_res.to_csv(f'{DRIVE_DIR}/results/eval_results.csv', index=False)

# ── Summary Table ──
summary = df_res[df_res['temperature']==0.0].groupby('split').agg(
    N=('output_correct','count'),
    OA_pct=('output_correct', lambda x: f"{x.mean()*100:.1f}%"),
    Avg_Lev=('levenshtein', lambda x: f"{x.mean():.1f}"),
    Avg_FES=('first_error_step', lambda x: f"{x.mean():.1f}"),
    Avg_NED=('norm_edit_dist', lambda x: f"{x.mean():.3f}")
)
print('══════════════ EVALUATION RESULTS ══════════════')
print(summary.to_string())

# OOD Generalisation Gap
oa_indist = df_res[(df_res['split']=='test_indist') & (df_res['temperature']==0.0)]['output_correct'].mean()*100
oa_ood    = df_res[(df_res['split']=='test_ood')    & (df_res['temperature']==0.0)]['output_correct'].mean()*100
print(f'\nOOD Generalisation Gap (RQ2): {oa_indist:.1f}% - {oa_ood:.1f}% = {oa_indist-oa_ood:.1f}%')

# ── Plot Grid ──
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    'Output Accuracy by Split', 'OA by Complexity',
    'Levenshtein Distance (State Fidelity)', 'Temperature Sensitivity'
])

# 1: OA by split
oa_split = df_res[df_res['temperature']==0.0].groupby('split')['output_correct'].mean()*100
colors_map = {'val':'#636EFA','test_indist':'#00CC96','test_ood':'#EF553B','test_long':'#FFA15A'}
fig.add_trace(go.Bar(
    x=oa_split.index, y=oa_split.values,
    marker_color=[colors_map.get(s,'#AB63FA') for s in oa_split.index],
    showlegend=False), row=1, col=1)

# 2: OA by complexity
oa_comp = df_res[df_res['temperature']==0.0].groupby('complexity')['output_correct'].mean()*100
fig.add_trace(go.Bar(
    x=[f'Level {c}' for c in oa_comp.index], y=oa_comp.values,
    marker_color=['#636EFA','#EF553B','#00CC96'], showlegend=False), row=1, col=2)

# 3: Levenshtein by split
lev_split = df_res[df_res['temperature']==0.0].groupby('split')['levenshtein'].mean()
fig.add_trace(go.Bar(
    x=lev_split.index, y=lev_split.values,
    marker_color=[colors_map.get(s,'#AB63FA') for s in lev_split.index],
    showlegend=False), row=2, col=1)

# 4: Temperature sensitivity (InDist: T=0 vs T=0.5)
t_data = df_res[df_res['split']=='test_indist'].groupby('temperature')['output_correct'].mean()*100
fig.add_trace(go.Bar(
    x=[f'T={t}' for t in t_data.index], y=t_data.values,
    marker_color=['#00CC96','#EF553B'], showlegend=False), row=2, col=2)

fig.update_layout(title='Evaluation Results — Phase 1 World Model', template='plotly_dark', height=700)
fig.update_yaxes(title_text='Accuracy (%)', row=1, col=1)
fig.update_yaxes(title_text='Accuracy (%)', row=1, col=2)
fig.update_yaxes(title_text='Edit Distance', row=2, col=1)
fig.update_yaxes(title_text='Accuracy (%)', row=2, col=2)
fig.show()

# Failure analysis: most common error types
wrong = df_res[(df_res['output_correct']==False) & (df_res['temperature']==0.0)]
print(f'\n🔍 Failures at T=0: {len(wrong)}/{len(df_res[df_res["temperature"]==0.0])} total')
print('Sample wrong outputs:')
for _, row in wrong.head(5).iterrows():
    print(f"  pred={repr(row['pred_output'][:20])} | true={repr(row['true_output'])} | complexity={row['complexity']}")

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 4 — RL Reward Simulation (GRPO-style)
# Demonstrates the trace-correctness reward signal that
# would drive a full GRPO training loop.
# ════════════════════════════════════════════════════════════
import random

# Keep model in inference mode from Stage 3

def compute_reward(generated_text, ground_truth_record):
    """
    GRPO-style reward: R ∈ [0, 1]
    = 0.4 × (output correct) + 0.6 × (normalised trace similarity)

    The trace reward rewards partial correctness — a model that gets
    the first 5 steps right before diverging gets more signal than
    one that is wrong from step 1.
    """
    pred_output = extract_output(generated_text)
    pred_trace  = extract_trace(generated_text)
    true_trace  = ground_truth_record['execution_trace']

    output_bonus = 1.0 if pred_output.strip() == ground_truth_record['output'].strip() else 0.0
    max_len  = max(len(pred_trace), len(true_trace), 1)
    lev_sim  = 1.0 - min(Levenshtein.distance(pred_trace, true_trace) / max_len, 1.0)
    fes      = first_error_step(pred_trace, true_trace)
    trace_steps = len([l for l in true_trace.split('\n') if 'var_' in l])
    fes_pct  = fes / max(trace_steps, 1)   # How far through trace before first error

    reward = 0.35 * output_bonus + 0.40 * lev_sim + 0.25 * fes_pct
    return reward, {'output_bonus': output_bonus, 'lev_sim': lev_sim, 'fes_pct': fes_pct}

# Rollout over 30 random programs (T=0.8 for diversity)
rl_batch = random.sample([r for r in records if r['split'] in ['train','val']], 30)
rl_results = []

print('Running RL rollouts...')
for i, r in enumerate(rl_batch):
    prompt = ALPACA_TEMPLATE.format(
        instruction=SYSTEM_PROMPT,
        input=f"<code>\n{r['code']}\n</code>",
        output=''
    )
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN).to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=300, temperature=0.8,
                              do_sample=True, pad_token_id=tokenizer.eos_token_id)
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    reward, breakdown = compute_reward(gen, r)
    rl_results.append({'complexity': r['complexity'], 'reward': reward, **breakdown})
    print(f'  [{i+1}/30] complexity={r["complexity"]} reward={reward:.3f}')

df_rl = pd.DataFrame(rl_results)
print(f'\n""" Reward Statistics """')
print(df_rl[['reward','output_bonus','lev_sim','fes_pct']].describe().round(3))

# Visualize reward breakdown
fig = make_subplots(rows=1, cols=2, subplot_titles=['Reward Distribution','Reward Component Breakdown'])
for comp, color in [(1,'#636EFA'),(2,'#EF553B'),(3,'#00CC96')]:
    sub = df_rl[df_rl['complexity']==comp]['reward']
    if len(sub): fig.add_trace(go.Histogram(x=sub, name=f'C{comp}', marker_color=color, opacity=0.7, nbinsx=15), row=1, col=1)

components = ['output_bonus', 'lev_sim', 'fes_pct']
means = [df_rl[c].mean() for c in components]
weights = [0.35, 0.40, 0.25]
fig.add_trace(go.Bar(
    x=['Output Accuracy (×0.35)', 'Trace Similarity (×0.40)', 'Steps Correct (×0.25)'],
    y=[m*w for m, w in zip(means, weights)],
    marker_color=['#636EFA','#EF553B','#00CC96'], showlegend=False), row=1, col=2)

fig.update_layout(title='RL Reward Signal Analysis (GRPO Surrogate)', template='plotly_dark', barmode='overlay')
fig.show()
df_rl.to_csv(f'{DRIVE_DIR}/results/rl_rewards.csv', index=False)

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 5A — MechInterp Setup: Merge LoRA, Load Interp Model
# We need a standard HF model (no PEFT wrapper) for hook access
# ════════════════════════════════════════════════════════════
from transformers import AutoModelForCausalLM

MERGED_PATH = f'{DRIVE_DIR}/checkpoints/qwen1.5b_condB_merged'

if not os.path.exists(f'{MERGED_PATH}/config.json'):
    print('Merging LoRA adapters into base weights...')
    merged = model.merge_and_unload()   # Unsloth: fuse LoRA into weights
    merged.save_pretrained(MERGED_PATH)
    tokenizer.save_pretrained(MERGED_PATH)
    del merged
    torch.cuda.empty_cache()
    print(f'✅ Merged model saved to {MERGED_PATH}')
else:
    print(f'✅ Merged model already on Drive, loading...')

# Load merged model in fp16 for activation access
interp_model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH, torch_dtype=torch.float16, device_map='auto')
interp_model.eval()
N_LAYERS = len(interp_model.model.layers)

# ─── Shared hook infrastructure ───────────────────────────
# A single dict stores activations; hooks write to it.
# This is shared across 5A, 5B, 5C — it avoids the bug
# where make_hook is defined in one cell but used in another.
_resid_cache = {}  # {layer_idx: tensor}

def run_with_hooks(prompt_str, max_len=MAX_SEQ_LEN):
    """Forward pass that captures residual stream at every layer."""
    global _resid_cache
    _resid_cache.clear()
    inputs = tokenizer(prompt_str, return_tensors='pt', truncation=True,
                       max_length=max_len).to('cuda')
    hooks = []
    for i, layer in enumerate(interp_model.model.layers):
        def make_hook(idx):
            def fn(module, inp, out):
                _resid_cache[idx] = out[0].detach().cpu().float()
            return fn
        hooks.append(layer.register_forward_hook(make_hook(i)))
    with torch.no_grad():
        output = interp_model(**inputs)
    for h in hooks: h.remove()
    return inputs, output

print(f'✅ Interpretability model loaded | Layers: {N_LAYERS} | d_model: {interp_model.config.hidden_size}')
print(f'   VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 5B — Logit Lens + Linear Probe
# What is the model thinking at each layer? What is encoded?
# ════════════════════════════════════════════════════════════
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
import numpy as np

# ── Pick an example trace for the Logit Lens ──
lens_sample = [r for r in records if r['complexity']==2 and r['split']=='test_indist'][0]
LENS_PROMPT  = f"<code>\n{lens_sample['code']}\n</code>\n\n<execution_trace>\n"

inputs_ll, _ = run_with_hooks(LENS_PROMPT)
n_tokens_ll  = inputs_ll['input_ids'].shape[1]

# Apply final LayerNorm + unembed to each layer's last-token residual
W_U      = interp_model.lm_head.weight.cpu().float()   # (vocab, d_model)
norm_fn  = interp_model.model.norm

top1_tokens, top1_probs, top5_tokens = [], [], []
for layer_idx in range(N_LAYERS):
    resid = _resid_cache[layer_idx][0, -1, :].unsqueeze(0).unsqueeze(0).to('cuda')
    normed = norm_fn(resid).squeeze().cpu().float()
    logits = W_U @ normed     # (vocab,)
    probs  = torch.softmax(logits, dim=-1)
    top_ids = probs.topk(5).indices
    top1_tokens.append(tokenizer.decode([top_ids[0].item()]))
    top1_probs.append(probs[top_ids[0]].item())
    top5_tokens.append([tokenizer.decode([t.item()]) for t in top_ids])

# ── Logit Lens Plot ──
fig_ll = go.Figure()
fig_ll.add_trace(go.Scatter(
    x=list(range(N_LAYERS)), y=top1_probs,
    mode='lines+markers', text=top1_tokens,
    hovertemplate='Layer %{x}<br>Token: <b>%{text}</b><br>Prob: %{y:.3f}',
    line=dict(color='#AB63FA', width=2), marker=dict(size=6)
))
fig_ll.update_layout(
    title='🔭 Logit Lens — Model Prediction Across Layers<br><sup>Higher prob = more confident early commitment to a token</sup>',
    xaxis_title='Transformer Layer', yaxis_title='Top-1 Token Probability',
    template='plotly_dark'
)
fig_ll.show()

# ── Linear Probe: var_0 value in residual stream ──
print('\nRunning linear probes across all layers...')
probe_recs = [r for r in records if r['split'] == 'test_indist'][:60]

def parse_final_var0(trace_str):
    """Get the last value assigned to var_0 in the trace."""
    vals = re.findall(r'var_0=([\-0-9]+)', trace_str)
    if not vals: return None
    try: return int(vals[-1])
    except: return None

X_all  = {i: [] for i in range(N_LAYERS)}
y_all  = []
for r in probe_recs:
    label = parse_final_var0(r['execution_trace'])
    if label is None: continue
    probe_text = f"<code>\n{r['code']}\n</code>\n\n{r['execution_trace']}"
    run_with_hooks(probe_text)  # fills _resid_cache
    for i in range(N_LAYERS):
        X_all[i].append(_resid_cache[i][0, -1, :].numpy())
    y_all.append(float(label))

r2_scores = []
for layer_idx in range(N_LAYERS):
    X = np.array(X_all[layer_idx])
    y = np.array(y_all)
    if len(X) < 10: r2_scores.append(0.0); continue
    X_sc = StandardScaler().fit_transform(X)
    cv_r2 = cross_val_score(Ridge(alpha=1.0), X_sc, y, cv=3, scoring='r2').mean()
    r2_scores.append(float(max(cv_r2, -0.1)))  # clip extreme negatives

# ── Linear Probe Plot ──
fig_probe = go.Figure()
fig_probe.add_trace(go.Scatter(
    x=list(range(N_LAYERS)), y=r2_scores,
    mode='lines+markers', fill='tozeroy', fillcolor='rgba(99,110,250,0.15)',
    line=dict(color='#636EFA', width=2),
    hovertemplate='Layer %{x}<br>R²=%{y:.3f}'
))
fig_probe.add_hline(y=0, line_dash='dash', line_color='red',
                    annotation_text='R²=0 (baseline, no info)', annotation_position='top right')
best_layer = int(np.argmax(r2_scores))
fig_probe.add_vline(x=best_layer, line_dash='dot', line_color='yellow',
                    annotation_text=f'Peak at L{best_layer}')
fig_probe.update_layout(
    title='📊 Linear Probe: var_0 State Encoded in Residual Stream<br><sup>R²>0 = model encodes variable value as a linear feature</sup>',
    xaxis_title='Layer', yaxis_title='Cross-val R² Score',
    template='plotly_dark'
)
fig_probe.show()
pd.DataFrame({'layer': range(N_LAYERS), 'r2': r2_scores}).to_csv(
    f'{DRIVE_DIR}/interp/probe_r2_by_layer.csv', index=False)
print(f'\n✅ Best probe R²={max(r2_scores):.3f} at Layer {best_layer}/{N_LAYERS-1}')
print(f'   (R²>0 = world model evidence; R²~0 = no state encoding)')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 5C — Activation Patching (Causal Tracing)
# Corrupt one variable's value; patch activations layer×token;
# measure causal effect on the model's output prediction.
# ════════════════════════════════════════════════════════════
# ─── Pick a clean/corrupted pair ──────────────────────────
patch_sample = [r for r in records if r['complexity']==1 and r['split']=='test_indist'][0]

# Clean: original code, predicting the correct output
clean_prompt = f"<code>\n{patch_sample['code']}\n</code>\n\n<execution_trace>\n"

# Corrupted: swap the first numeric literal in the code (change var_0's initial value)
original_val = re.search(r'var_0 = ([\-]?\d+)', patch_sample['code'])
if original_val:
    corrupt_code = patch_sample['code'].replace(
        original_val.group(0), f"var_0 = {int(original_val.group(1))+99}", 1)
else:
    corrupt_code = patch_sample['code']  # fallback: same code
corrupt_prompt = f"<code>\n{corrupt_code}\n</code>\n\n<execution_trace>\n"

print('Clean code snippet:')
print(patch_sample['code'][:100])
print('\nCorrupted code snippet:')
print(corrupt_code[:100])

# Get target token ID for the correct output
target_token_str = str(patch_sample['output']).strip()
target_tok_ids   = tokenizer(target_token_str, add_special_tokens=False).input_ids
target_tok_id    = target_tok_ids[-1] if target_tok_ids else 0

def get_logit(prompt_str, tok_id):
    _, out = run_with_hooks(prompt_str)
    return out.logits[0, -1, tok_id].item()

clean_logit   = get_logit(clean_prompt, target_tok_id)
corrupt_logit = get_logit(corrupt_prompt, target_tok_id)
baseline_diff = clean_logit - corrupt_logit
print(f'\nClean→correct token logit: {clean_logit:.3f}')
print(f'Corrupt→correct token logit: {corrupt_logit:.3f}')
print(f'Baseline difference: {baseline_diff:.3f}')

# ─── Cache corrupted activations ──────────────────────────
corrupt_inputs, _ = run_with_hooks(corrupt_prompt)
corrupt_cache     = {i: _resid_cache[i].clone() for i in range(N_LAYERS)}

# ─── Patch sweep ──────────────────────────────────────────
clean_inputs = tokenizer(clean_prompt, return_tensors='pt', truncation=True,
                          max_length=MAX_SEQ_LEN).to('cuda')
n_patch_pos  = min(clean_inputs['input_ids'].shape[1],
                   corrupt_inputs['input_ids'].shape[1], 25)

effect_matrix = np.zeros((N_LAYERS, n_patch_pos))
print(f'\nRunning {N_LAYERS}×{n_patch_pos} patch sweep...')

for layer_idx in range(N_LAYERS):
    if layer_idx % 5 == 0: print(f'  Layer {layer_idx}/{N_LAYERS}...')
    for pos in range(n_patch_pos):
        # Install a patch hook at this (layer, pos)
        def make_patch_hook(lidx, position):
            corr = corrupt_cache[lidx].to('cuda')
            def fn(module, inp, out):
                o = list(out)
                o[0] = o[0].clone()
                safe_pos = min(position, corr.shape[1]-1)
                o[0][:, position, :] = corr[:, safe_pos, :]
                return tuple(o)
            return fn
        h = interp_model.model.layers[layer_idx].register_forward_hook(
                make_patch_hook(layer_idx, pos))
        with torch.no_grad():
            patched_logit = interp_model(**clean_inputs).logits[0, -1, target_tok_id].item()
        h.remove()
        effect = (clean_logit - patched_logit) / (abs(baseline_diff) + 1e-8)
        effect_matrix[layer_idx, pos] = effect

# ─── Visualise ────────────────────────────────────────────
token_labels = [repr(tokenizer.decode([t]))
                for t in clean_inputs['input_ids'][0, :n_patch_pos].tolist()]
fig_patch = go.Figure(data=go.Heatmap(
    z=effect_matrix, x=token_labels,
    y=[f'L{i}' for i in range(N_LAYERS)],
    colorscale='RdBu_r', zmid=0, zmin=-1, zmax=1,
    colorbar=dict(title='Effect<br>(+1=full<br>corruption)')
))
fig_patch.update_layout(
    title='🗺️ Activation Patching: Which Layer×Token Causally Determines Output?<br>'
          '<sup>Red = patching here strongly corrupts output | Blue = no effect</sup>',
    xaxis_title='Token (position in clean prompt)',
    yaxis_title='Transformer Layer',
    template='plotly_dark', height=600
)
fig_patch.show()
fig_patch.write_html(f'{DRIVE_DIR}/interp/patching_heatmap.html')

# Find hotspot
hotspot  = np.unravel_index(effect_matrix.argmax(), effect_matrix.shape)
print(f'\n🔥 Hotspot: Layer {hotspot[0]}, Token {token_labels[hotspot[1]]} (effect={effect_matrix[hotspot]:.3f})')
print(f'   Sharp hotspot → causal world model evidence')
print(f'   Diffuse/flat  → memorisation or pure pattern matching')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 6 — Save Everything to Drive & Final Report
# ════════════════════════════════════════════════════════════
import json, datetime

# ── Persist model & results ──
trainer.save_model(f'{DRIVE_DIR}/checkpoints/qwen1.5b_condB_final')
tokenizer.save_pretrained(f'{DRIVE_DIR}/checkpoints/qwen1.5b_condB_final')
df_res.to_csv(f'{DRIVE_DIR}/results/eval_results.csv', index=False)

# ── Compute final numbers ──
oa_val    = df_res[(df_res['split']=='val')         & (df_res['temperature']==0.0)]['output_correct'].mean()*100
oa_indist = df_res[(df_res['split']=='test_indist') & (df_res['temperature']==0.0)]['output_correct'].mean()*100
oa_ood    = df_res[(df_res['split']=='test_ood')    & (df_res['temperature']==0.0)]['output_correct'].mean()*100
ood_gap   = oa_indist - oa_ood
avg_lev   = df_res[(df_res['split']=='test_indist') & (df_res['temperature']==0.0)]['levenshtein'].mean()
avg_fes   = df_res[(df_res['split']=='test_indist') & (df_res['temperature']==0.0)]['first_error_step'].mean()
temp_drop = oa_indist - df_res[(df_res['split']=='test_indist') & (df_res['temperature']==0.5)]['output_correct'].mean()*100
best_r2   = max(r2_scores)
best_r2_l = int(np.argmax(r2_scores))

# Write structured results JSON
results_summary = {
    'run_date':       str(datetime.datetime.now()),
    'model':          MODEL_NAME,
    'condition':      CONDITION,
    'final_loss':     final_loss,
    'oa_val':         round(oa_val, 2),
    'oa_test_indist': round(oa_indist, 2),
    'oa_test_ood':    round(oa_ood, 2),
    'ood_gap':        round(ood_gap, 2),
    'avg_levenshtein':round(avg_lev, 1),
    'avg_fes':        round(avg_fes, 1),
    'temp_drop_0_to_0.5': round(temp_drop, 2),  # Temp sensitivity
    'best_probe_r2':  round(best_r2, 4),
    'best_probe_layer': best_r2_l,
    'probe_r2_by_layer': r2_scores,
}
with open(f'{DRIVE_DIR}/results/results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

try: wandb.log(results_summary); wandb.finish()
except: pass

# ── Print research report ──
print()
print('╔══════════════════════════════════════════════════════════════╗')
print('║          LM WORLD MODEL — PHASE 1 RESULTS                   ║')
print('║   Qwen2.5-1.5B · Condition B · Full Trace Supervision        ║')
print('╠════════════════════════════════╦═════════════════════════════╣')
print(f'║  Output Acc (Val)              ║  {oa_val:5.1f}%                  ║')
print(f'║  Output Acc (InDist, T=0)      ║  {oa_indist:5.1f}%                  ║')
print(f'║  Output Acc (OOD, T=0)         ║  {oa_ood:5.1f}%                  ║')
print(f'║  OOD Generalisation Gap        ║  {ood_gap:+5.1f}%                  ║')
print(f'║  Avg Levenshtein (InDist)      ║  {avg_lev:7.1f}                ║')
print(f'║  Avg First-Error Step          ║  {avg_fes:7.1f}                ║')
print(f'║  Accuracy drop (T 0→0.5)       ║  {temp_drop:+5.1f}% (stability)     ║')
print(f'║  Best Probe R² (var_0)         ║  {best_r2:5.3f} (Layer {best_r2_l:2d})       ║')
print('╠════════════════════════════════╩═════════════════════════════╣')
print(f'║  H0 REJECT if OA(OOD) >> chance (~{100/max(len(set(r["output"] for r in ood_recs)), 1):.0f}%)')
print(f'║  World model evidence if Probe R² > 0 ✓ (={best_r2:.3f})')
print('╚══════════════════════════════════════════════════════════════╝')
print(f'\n📁 All results saved to: {DRIVE_DIR}/results/')